In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import wget
import os
import scgen
from scgen.file_utils import ensure_dir_for_file
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80)  # low dpi (dots per inch) yields small inline figures
sc.logging.print_versions()

/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
2026-02-17 02:45:21.270641: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-17 02:45:21.372118: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH:

Instructions for updating:
non-resource variables are not supported in the long term


Package,Version
wget,3.2
Component,Info
Python,"3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]"
OS,Linux-6.1.159-181.297.amzn2023.x86_64-x86_64-with-glibc2.35
CPU,"16 logical CPU cores, x86_64"
GPU,"ID: 0, NVIDIA L40S, Driver: 580.126.09, Memory: 46068 MiB"
Updated,2026-02-17 02:45
Dependency,Version
llvmlite,0.46.0
h5py,3.15.1


In [2]:
%load_ext rpy2.ipython

In [3]:
%%R
library(Seurat)
packageVersion("Seurat")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    [1] ‘5.4.0’


Loading required package: SeuratObject
Loading required package: sp

Attaching package: ‘SeuratObject’

The following objects are masked from ‘package:base’:

    intersect, t



In [4]:
train_path = "../data/pancreas.h5ad"
if os.path.isfile(train_path):
    adata = scgen.load_file(train_path)
else:
    train_url = "https://www.dropbox.com/s/zvmt8oxhfksumw2/pancreas.h5ad?dl=1"
    t_dl = wget.download(train_url, train_path)
    adata = scgen.load_file(train_path)

adata = anndata.AnnData(X=np.expm1(adata.raw.X), var=adata.raw.var, obs=adata.obs)
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
filter_result = sc.pp.filter_genes_dispersion(
    adata.X, min_mean=0.0125, max_mean=2.5, min_disp=0.7)
adata = adata[:, filter_result.gene_subset]
sc.pp.log1p(adata)

df1 = pd.DataFrame(data=adata[adata.obs['sample']=='Baron'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Baron'].var_names,
                  columns=adata[adata.obs['sample']=='Baron'].obs_names)

df2 = pd.DataFrame(data=adata[adata.obs['sample']=='Muraro'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Muraro'].var_names,
                  columns=adata[adata.obs['sample']=='Muraro'].obs_names)

df3 = pd.DataFrame(data=adata[adata.obs['sample']=='Segerstolpe'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Segerstolpe'].var_names,
                  columns=adata[adata.obs['sample']=='Segerstolpe'].obs_names)

df4 = pd.DataFrame(data=adata[adata.obs['sample']=='Wang'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Wang'].var_names,
                  columns=adata[adata.obs['sample']=='Wang'].obs_names)

/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


normalizing by total count per cell


/tmp/ipykernel_12050/1329661529.py:10: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:590: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(


    finished (0:00:00): normalized adata.X and added
    'n_counts', counts per cell before normalization (adata.obs)
extracting highly variable genes


/tmp/ipykernel_12050/1329661529.py:11: FutureWarning: Use sc.pp.highly_variable_genes instead
  filter_result = sc.pp.filter_genes_dispersion(


    finished (0:00:02)


/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:412: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [5]:
%%R -i df1 -i df2 -i df3 -i df4 -o cca
suppressPackageStartupMessages(library(Seurat))

mk <- function(m, batch_name) {
    # formerly: sdf1 = CreateSeuratObject(df1)
    obj <- CreateSeuratObject(counts = m, project = batch_name)
    # formerly: sdf1@meta.data$batch = batch_name
    obj$batch <- batch_name
    # formerly: sdf1 = ScaleData(sdf1)
    obj <- NormalizeData(obj, normalization.method = "LogNormalize", scale.factor = 1e4)
    # formerly: sdf1@var.genes = rownames(df1)
    obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 2000)
    obj
}

objs <- list (
    mk(as.matrix(df1), "1"),
    mk(as.matrix(df2), "2"),
    mk(as.matrix(df3), "3"),
    mk(as.matrix(df4), "4")
)

t1 = Sys.time()
# formeerly: RunMultiCCA(list(sdf1, sdf2, sdf3, sdf4), genes.use=rownames(df1), num.ccs=50) & AlignSubspace(srat, reduction.type='cca', grouping.var='batch', dims.align=1:50)
features <- SelectIntegrationFeatures(object.list = objs, nfeatures = 2000)
anchors <- FindIntegrationAnchors(object.list = objs, anchor.features = features, reduction = "cca", dims = 1:50)
integrated <- IntegrateData(anchorset = anchors, dims = 1:50)

DefaultAssay(integrated) <- "integrated"
integrated <- ScaleData(integrated)
integrated <- RunPCA(integrated, npcs = 50)
t2 = Sys.time()
print(t2-t1)
cca <- as.data.frame(Embeddings(integrated, reduction = "pca")[, 1:50])

  |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=01s  
  |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=03m 21s
Time difference of 3.929465 mins


Normalizing layer: counts
Performing log-normalization
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Finding variable features for layer counts
Calculating gene variances
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Calculating feature variances of standardized and clipped values
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Normalizing layer: counts
Performing log-normalization
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Finding variable features for layer counts
Calculating gene variances
0%   10   20   30   40   50   60   70   80   90  

In [6]:
adata_cca = adata.copy()
adata_cca.obsm['X_pca'] = cca.values

In [7]:
adata_cca.write(ensure_dir_for_file("cca.h5ad"))